# Attention Motif Discovery

Discovers recurring attention patterns (motifs) across heads and layers using SVD-based factorization. Goes beyond head classification to find abstract structural patterns.

**References:**
- Olsson et al. (2022) "In-context Learning and Induction Heads"
- Voita et al. (2019) "Analyzing Multi-Head Self-Attention"

## Why This Matters

Attention motif discovery uses SVD to extract recurring attention patterns across different inputs. These motifs — like 'attend to previous token' or 'attend to separator' — represent the functional building blocks from which complex behaviors are composed.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_motif_discovery import (
    extract_attention_motifs,
    motif_prevalence_analysis,
    motif_input_dependency,
    motif_function_inference,
    motif_diversity_by_layer,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Extract motifs
motifs = extract_attention_motifs(model, tokens, n_motifs=4)
print(f"Found {motifs['n_motifs_found']} motifs")
print(f"Assignments:\n{motifs['head_motif_assignments']}")

In [ ]:
# 2. Prevalence
prev = motif_prevalence_analysis(model, tokens, n_motifs=4)
print(f"Motif counts: {prev['motif_counts']}")
print(f"Dominant motif: {prev['dominant_motif']}")
print(f"Diversity: {prev['motif_diversity']:.4f}")

In [ ]:
# 3. Input dependency for a head
dep = motif_input_dependency(model, tokens, layer=0, head=0)
print(f"Diagonal strength: {dep['diagonal_strength']:.4f}")
print(f"Recency strength: {dep['recency_strength']:.4f}")
print(f"Uniformity: {dep['uniformity']:.4f}")

In [ ]:
# 4. Diversity by layer
div = motif_diversity_by_layer(model, tokens)
for l in range(cfg.n_layers):
    print(f"Layer {l}: diversity={div['layer_diversity'][l]:.4f}, "
          f"similarity={div['head_similarities'][l]:.4f}")